# Interpret a vision-model score

**Purpose.** Model vision-classifier outputs using measured morphology and intensity features to identify associated quantitative attributes.

**Recommended use.** Use to relate image-model predictions to interpretable cellular measurements while retaining grouped validation.

**Primary outputs.** Surrogate-model fidelity metrics and ranked feature explanations.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.submodules.interpret_vision_model`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.interpret_vision_model)

```python
interpret_vision_model(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.submodules import interpret_vision_model

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.submodules.interpret_vision_model`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.interpret_vision_model)

> This function does not yet expose a settings factory. The list below is recovered from direct settings access in its source, so a key used only on a dynamic branch may be absent.


#### Paths

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.

#### General

- **`channels`** *(optional)* — (list of int) - Zero-indexed image channels kept in merged/*.npy and measured by measure_crop; each entry produces its own &lt;object&gt;_channel_&lt;n&gt;_* intensity columns. The list length fixes where masks land, so cell/nucleus/pathogen_mask_dim must shift if you change it. Preprocessing silently resets it to range(n) when it does not match the number of channel folders found. Default [0,1,2,3].

#### Measurements

- **`tables`** *(optional)* — (list) - Measurement tables read from each plate's database and merged into one analysis frame. Only 'cell', 'nucleus', 'pathogen', 'cytoplasm' and 'png_list' are actually merged. Any other name -- INCLUDING 'organelle', which the measure step does write -- is loaded and then dropped, so asking for it costs time and returns nothing, with no warning that the table you wanted is missing from the result. Default ['cell', 'nucleus', 'pathogen', 'cytoplasm'].
- **`nuclei_limit`** *(optional)* — (int, bool, or None) - Cap on nuclei per cell, applied when the per-object tables are merged. None disables the filter, True keeps only single-nucleus cells, and an integer N keeps cells with N or fewer. Cells over the cap are dropped from the merged table entirely. Do NOT pass False: it is read as 0 and removes every cell, leaving an empty analysis rather than an error. Default None.
- **`pathogen_limit`** *(optional)* — (int, bool, or None) - Maximum pathogens per cell. True or 1 = single pathogen only; None or False = no limit; int = custom limit. Default varies by module (1, 3, 10 or 1000 depending on the factory that fills it), so check the module's own settings rather than assuming one value.

#### Model Evaluation

- **`score_column`** *(optional)* — (str) - Which column of the prediction CSV holds the CNN score that Explain CV and the hit-investigation montages read. The regression module no longer has this setting: it fits dependent_variable and simulates the minimum cell count on that same column, so one measurement cannot be named two ways there. Default 'cv_predictions'.

#### Machine Learning Model and Features

- **`top_features`** *(optional)* — (int) - Feature cap in the ML screen analysis: how many rows the feature-importance and permutation-importance bar plots show, and how many top-ranked features the SHAP refit and its summary plot use. It is also the k of the SelectKBest pruning applied before the model is fitted, but only when prune_features is True - with prune_features at its default False the classifier trains on every feature and this is reporting/SHAP scope only. Raise for a fuller picture, lower for readable plots. Default 30.

#### Advanced

- **`n_jobs`** *(optional)* — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.
- **`save`** *(optional)* — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.

#### Additional settings

- **`feature_importance`** *(optional)* — (bool) - Fit a random forest against the score column and plot its impurity-based importances. Fast and always available, but biased toward high-cardinality and correlated features, so read the result as a shortlist rather than a ranking. Turning it off skips that plot and its two grouped-by-compartment and grouped-by-channel companions. Default True.
- **`include_all`** *(optional)* — (bool) - When grouping feature importances by compartment and by channel, also emit an 'all' row totalling the features that belong to no single compartment or channel. With it off those features are simply absent from the grouped plots, so the bars no longer sum to the whole and a large shared contribution is invisible. Default False.
- **`permutation_importance`** *(optional)* — (bool) - Re-score the fitted forest with each feature shuffled in turn, ten repeats, and rank by how far the score falls. Much slower than feature_importance and far more trustworthy, because a feature that shuffles harmlessly was not being used. Turn it on when a shortlist has to become a claim. Default False.
- **`scores`** *(optional)* — (str, path) - CSV of per-object model scores to interpret, joined to the measurements on plateID, rowID, columnID, fieldID and object_label. This is a classification run's output, and the interpretation explains THESE scores - a CSV from a different model or plate yields a confident explanation of the wrong thing rather than an error. Default None.
- **`shap`** *(optional)* — (bool) - Compute SHAP values, which attribute each individual prediction to each feature instead of ranking features overall. It is the only one of the three that can explain a single object, and by far the slowest - see shap_sample before enabling it on a full plate. Default False.
- **`shap_sample`** *(optional)* — (bool) - Run SHAP on a subsample rather than every object. SHAP cost grows with the row count, so on a full plate this is the difference between minutes and hours, and the feature ranking is stable long before the individual values are. Turn it off only when a specific object's attribution has to be exact. Default True.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Paths
    # Required settings
    'src': None,

    # General
    # Optional settings
    'channels': None,

    # Measurements
    # Optional settings
    'tables': None,
    'nuclei_limit': None,
    'pathogen_limit': None,

    # Model Evaluation
    # Optional settings
    'score_column': None,

    # Machine Learning Model and Features
    # Optional settings
    'top_features': None,

    # Advanced
    # Optional settings
    'n_jobs': None,
    'save': None,

    # Additional settings
    # Optional settings
    'feature_importance': None,
    'include_all': None,
    'permutation_importance': None,
    'scores': None,
    'shap': None,
    'shap_sample': None,
}

In [ ]:
interpret_vision_model(settings)

## Outputs and next steps

Surrogate-model fidelity metrics and ranked feature explanations.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)